# Pipeline do projeto

In [3]:
import sys
from pathlib import Path

# bibliotecas gerais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# classes do projeto
from feature_engineering import EngenhariaFeaturesTelco
from baseline_training import BaselineTrainer
from mlp_training import MLPTrainer
from otimizacao import OtimizadorMLP
from ensemble import EnsembleMLP

c:\Users\gui08\OneDrive\Documentos\Tech_challenge_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## EDA

In [4]:
caminho_dados = Path("data\\Telco_customer_churn.xlsx")
coluna_alvo_binaria = "target"
coluna_score = "score"

df = pd.read_excel(caminho_dados)


## Engenharia de Features

In [6]:
feature_engineering = EngenhariaFeaturesTelco()
df_modelo = feature_engineering.processar(
    caminho_entrada=caminho_dados,
    caminho_saida=Path("data/processed/telco_feature_engineered.csv")
)

## Treinamento Dummy Baseline

In [7]:
baseline = BaselineTrainer(
    input_path='data/processed/telco_feature_engineered.csv',
    target='Churn Value',
    test_size=0.2,
    random_seed=42,
)
baseline_payload = baseline.executar()

In [10]:
results = baseline_payload['results']  # dict com keys 'dummy_classifier' e 'logistic_regression'
metrics_df = pd.DataFrame(results).T
display(metrics_df)

,accuracy,precision,recall,f1,roc_auc
dummy_classifier,0.734564,0.000000,0.000000,0.000000,0.50000
logistic_regression,0.743790,0.511424,0.778075,0.617179,0.84772


## Treinamento MLP

In [11]:
mlp_binary = MLPTrainer(
    input_path='data/processed/telco_feature_engineered.csv',
    task='binary',
    experiment_name='telco_churn_mlp',
    run_name='mlp_binary',
    output_dir='results/mlp',
    test_size=0.2,
    val_size=0.2,
    batch_size=64,
    hidden_sizes='128,64',
    dropout=0.2,
    learning_rate=1e-3,
    weight_decay=1e-5,
    epochs=50,
    patience=10,
    random_seed=42,
)
binary_results = mlp_binary.treinar_tarefa('binary')

2026/08/19 20:31:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/19 20:31:22 INFO mlflow.store.db.utils: Updating database tables
2026/08/19 20:32:20 INFO mlflow.tracking.fluent: Experiment with name 'telco_churn_mlp' does not exist. Creating a new experiment.


In [13]:
mlp_score = MLPTrainer(
    input_path='data/processed/telco_feature_engineered.csv',
    task='score',
    experiment_name='telco_churn_mlp',
    run_name='mlp_score',
    output_dir='results/mlp',
    test_size=0.2,
    val_size=0.2,
    batch_size=64,
    hidden_sizes='128,64',
    dropout=0.2,
    learning_rate=1e-3,
    weight_decay=1e-5,
    epochs=50,
    patience=10,
    random_seed=42,
)
score_results = mlp_score.treinar_tarefa('score')

In [ ]:
import joblib
objeto = joblib.load(Path('results//mlp//binary//scaler.joblib'))


<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\gui08\AppData\Local\Temp\ipykernel_16848\25188182.py:2: SyntaxWarning: invalid escape sequence '\m'
  objeto = joblib.load(Path('results\mlp\binary\scaler.joblib'))


OSError: [Errno 22] Invalid argument: 'results\\mlp\x08inary\\scaler.joblib'

## Métricas

In [ ]:
metricas = calcular_metricas_classificacao(
    y_binario=y_binario,
    y_score=y_score,
    pred_binario=pred_binario,
    pred_score=pred_score,
    pred_ensemble=pred_ensemble,
)

In [ ]:
print("\nMétricas finais:")
for nome, valor in metricas.items():
    print(f"{nome}: {valor:.4f}")

## Otimização

## Ensemble do Modelo final